In [1]:
%run start.py

Root set to: /home/bdudas/obesity_challange


In [2]:
import anndata as ad
import numpy as np
import torch
import torch.nn as nn
from src.data.vae_data import get_loaders
from src.data.perturbation_data import get_loaders
from omegaconf import OmegaConf
from src.models.transformerVAE import TransformerVAEEncoder, TransformerVAEDecoder,Transfomer_latent_Classifier


from src.models.vae_trainers import StateTrainer_latent

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
z_dim = 128
traning_config = OmegaConf.load("configs/traning.yaml")
encoder_config = OmegaConf.load("configs/encoder.yaml")
encoder_config.z_dim = z_dim

decoder_config = OmegaConf.load("configs/decoder.yaml")
decoder_config.z_dim = z_dim

classifier_config = OmegaConf.load("configs/classifier_latent.yaml")
classifier_config.z_dim = z_dim
trainer_config = OmegaConf.load("configs/trainer.yaml")


In [4]:

encoder = TransformerVAEEncoder(**encoder_config)
decoder = TransformerVAEDecoder(**decoder_config)
classifier = Transfomer_latent_Classifier(**classifier_config)

model = StateTrainer_latent(encoder,decoder,categorizer=classifier,**trainer_config)
cpkt_path = "misc/best_runs/latent_reg/checkpoints/epoch=19-step=5520.ckpt"
state_dict = torch.load(cpkt_path,weights_only=False)
model.load_state_dict(state_dict['state_dict'])

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


<All keys matched successfully>

In [8]:
trainloader, valloader, gene_to_idx, idx_to_gene = get_loaders("",batch_size=4)

In [9]:
batch = next(iter(valloader))

In [12]:
x, x_in, target_gene, state, input_state = batch

In [ ]:
#TODO: NEED TO TEST IF THIS WORKS

class PerturbationAdder(torch.nn.Module):
    def __init__(self, base_model,z_dim, pert_dim,inputLoader):
        super(PerturbationAdder, self).__init__()
        self.base_model.base_model = base_model
        self.pert_projection = nn.Linear(pert_dim, z_dim)
        self.cross_attention = nn.MultiheadAttention(embed_dim=z_dim, num_heads=8, batch_first=True)

        self.inputLoader = inputLoader
    def shared_step(self, batch, mode="Train"):
        Xs, x_in,target_gene, state, input_state = batch
        mu, logvar = self.base_model.encoder(Xs)
        logvar = torch.clamp(logvar, min=-6.0, max=2.0)
        z = self.base_model.reparameterize(mu, logvar)
        # Project input_state to match z dimension
        pert_proj = self.pert_projection(input_state)
        # Add a sequence dimension for attention mechanism
        z_seq = z.unsqueeze(1)  # Shape: [Batch, 1, z_dim]
        pert_seq = pert_proj.unsqueeze(1)  # Shape: [Batch, 1, z_dim]
        # Apply cross-attention
        z_attended, _ = self.cross_attention(z_seq, pert_seq, pert_seq)
        z = z_attended.squeeze(1)  # Remove sequence dimension
        Xs_hat = self.base_model.decoder(z)

        loss_recon = nn.functional.mse_loss(Xs_hat, Xs, reduction=self.base_model.reduction)
        loss_kld = self.base_model.compute_kl_loss(mu, logvar)
        loss_kld = torch.clamp(loss_kld, max=self.base_model.KLD_MAX)
        beta = self.base_model.kl_weight()

        if self.categorizer is not None:
            class_logits = self.categorizer(Xs_hat)
            loss_class = nn.functional.cross_entropy(class_logits, state)
        else:
            class_logits = None
            loss_class = torch.tensor(0.0, device=Xs.device)


        if state.shape == class_logits.shape:
            state_indices = torch.argmax(state, dim=1)
            # 2. If state is [Batch, 1] -> Squeeze to [Batch]
        elif state.ndim == 2 and state.shape[1] == 1:
            state_indices = state.squeeze(1)
        else:
            state_indices = state
        classWeight = self.classFactor 
        #self.classFactor_weight() if beta > 0.95 else 0.8
        total_loss = (self.reconFactor * loss_recon+ beta * loss_kld+ classWeight * loss_class)

        # Logging
        #self.logging_step(loss_recon, loss_kld, beta, loss_class, total_loss, class_logits, state_indices, mode=mode)

        return total_loss

StateTrainer_latent(
  (encoder): TransformerVAEEncoder(
    (token_proj): Linear(in_features=675, out_features=512, bias=True)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (mu): Linear(in_features=512, out_features=128, bias=True)
    (logvar): Linear(in_features=512, out_features=128, bias=True)
  )

In [ ]:
batch = next(iter(inputLoader))
X, target_gene, state = batch

In [18]:
state

tensor([[0., 0., 0., 1.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 0., 1.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 0., 1.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [1., 0., 0., 1.],
        [0., 0., 1., 0.],
        [1., 0., 0., 1.],
        [0., 0., 1., 0.],
        [1., 0., 0., 1.],
        [0., 0., 0., 1.],
        [0., 0., 1., 0.],
        [1., 0., 0., 1.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.],
        [0., 0., 1., 0.],
        [1., 0., 0., 1.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        [0.,